# Pipeline complet Deribit — BTC options de bout en bout (V2)

Ce notebook descend toute la pile de la plateforme, couche par couche, sur des **vraies options BTC
Deribit** (API publique, aucune authentification requise) :

```
Connexion  →  Discovery  →  Snapshot  →  Forwards  →  IV  →  Surface SVI  →  Pricing  →  Risk  →  Scénarios
```

**Mode `LIVE_DATA = True` (défaut)** — appelle l'API Deribit en temps réel.  
**Mode `LIVE_DATA = False`** — utilise des fixtures synthétiques BTC-like si Deribit est inaccessible.

Chaque cellule est indépendante : elle appelle uniquement les engines testés et n'ajoute aucune
logique métier ici.

Run : `uv run --group notebooks jupyter lab notebooks/demo_pipeline_deribit_v2.ipynb`

In [ ]:
# %% Chart config — define once, apply everywhere
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# ── Palette ──────────────────────────────────────────────────────────────────
C = {
    "blue":      "#2563EB",  # primary series / calls
    "teal":      "#0D9488",  # puts / complementary
    "violet":    "#7C3AED",  # gamma / vega / 3rd series
    "amber":     "#D97706",  # warning / stress
    "red":       "#DC2626",  # rejected / calendar violation
    "green":     "#16A34A",  # usable / pass
    "indigo":    "#4F46E5",  # 2nd expiry series
    "slate900":  "#0F172A",  # titles
    "slate600":  "#475569",  # axis labels
    "slate400":  "#94A3B8",  # gridlines / zero lines
    "slate100":  "#F1F5F9",  # panel backgrounds
    "white":     "#FFFFFF",
}

DISCRETE = [C["blue"], C["teal"], C["violet"], C["amber"], C["indigo"], "#0EA5E9", "#F59E0B"]
FONT     = "Inter, IBM Plex Sans, -apple-system, sans-serif"

# ── Surface 3D params ─────────────────────────────────────────────────────────
SURFACE_COLORSCALE = [
    [0.00, "#1E3A5F"],  # deep navy   — low vol
    [0.25, "#2563EB"],  # blue
    [0.50, "#0D9488"],  # teal        — mid vol
    [0.75, "#D97706"],  # amber
    [1.00, "#DC2626"],  # red         — high vol / skew
]
SURFACE_CAMERA = dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))
SURFACE_ASPECT = dict(x=1.5, y=1.2, z=0.7)  # flatten z — IV range narrow vs k/t footprint

# ── Plotly template ───────────────────────────────────────────────────────────
_axis = dict(
    showgrid=True, gridcolor=C["slate400"], gridwidth=0.5,
    zeroline=False, linecolor=C["slate400"], linewidth=1,
    ticks="outside", tickcolor=C["slate400"],
    tickfont=dict(size=11), title_font=dict(size=12, color=C["slate600"]),
)
_tmpl = go.layout.Template()
_tmpl.layout = go.Layout(
    font=dict(family=FONT, size=12, color=C["slate600"]),
    title_font=dict(family=FONT, size=15, color=C["slate900"]),
    paper_bgcolor=C["white"],
    plot_bgcolor=C["white"],
    colorway=DISCRETE,
    xaxis=_axis, yaxis=_axis,
    legend=dict(
        bgcolor="rgba(255,255,255,0.85)", bordercolor=C["slate400"], borderwidth=1,
        font=dict(size=11), tracegroupgap=4,
    ),
    margin=dict(l=64, r=24, t=56, b=56),
    hoverlabel=dict(
        bgcolor=C["slate900"], font=dict(color=C["white"], size=12, family=FONT),
        bordercolor=C["slate900"],
    ),
    hovermode="closest",
)
pio.templates["algotrading"] = _tmpl
pio.templates.default = "plotly_white+algotrading"

print("Chart config loaded — template: algotrading (Plotly V2)")

In [ ]:
# %% Setup — imports, chemins, flag live/synthetic
from __future__ import annotations

import math
from collections import defaultdict
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from pathlib import Path

import numpy as np

from algotrading.core.config import load_config
from algotrading.infra.forwards import ForwardParams, build_forward_curve
from algotrading.infra.iv import IvParams, solve_chain
from algotrading.infra.pricing import OptionStyle, PricingInput, PricingParams, price
from algotrading.infra.risk import (
    InstrumentAnalytics,
    Position,
    PositionSet,
    Scenario,
    ScenarioGrid,
    aggregate_risk,
    by_underlying,
    compute_risk_lines,
    run_grid,
)
from algotrading.infra.snapshots import MarketStateSnapshot, SnapshotParams, build_snapshot
from algotrading.infra.storage.events import RawMarketEvent
from algotrading.infra.surfaces import SurfaceConfig, build_surface
from algotrading.infra.surfaces.engine import SurfaceResult
from algotrading.infra.universe.contracts import OptionContract, Right, Underlying, instrument_key
from algotrading.infra.utils.daycount import DayCountConvention
from algotrading.infra_deribit.collectors.deribit_discovery import discover_instruments
from algotrading.infra_deribit.connectivity.deribit_transport import DeribitTransport


def _repo_root() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").exists() and (p / "packages").is_dir():
            return p
    raise FileNotFoundError("repo root non trouvé (pyproject.toml + packages/)")


REPO = _repo_root()
CONFIGS = REPO / "packages" / "infra" / "configs"

LIVE_DATA = True
CURRENCY  = "BTC"
MIN_DTE   = 7
MAX_DTE   = 120

print(f"Repo : {REPO}")
print(f"Mode : {'LIVE Deribit' if LIVE_DATA else 'SYNTHETIC fixtures'}")
print(f"Cible: {CURRENCY} options, {MIN_DTE}–{MAX_DTE} DTE")

## 1 · Connexion Deribit & prix index

In [ ]:
# %% Connexion + prix index
INDEX_NAME = f"{CURRENCY.lower()}_usd"

if LIVE_DATA:
    transport = DeribitTransport()
    index_data = transport.get("/public/get_index_price", {"index_name": INDEX_NAME})
    INDEX_PRICE: float = float(index_data["index_price"])
    print(f"✓ Connecté à Deribit — {CURRENCY}/USD index : ${INDEX_PRICE:,.0f}")
else:
    transport = None
    INDEX_PRICE = 65_000.0
    print(f"✓ Mode synthétique — {CURRENCY}/USD fictif : ${INDEX_PRICE:,.0f}")

## 2 · Découverte des instruments

In [ ]:
# %% Découverte instruments
if LIVE_DATA:
    contracts = discover_instruments(transport, CURRENCY, min_days=MIN_DTE, max_days=MAX_DTE)
else:
    today = date.today()
    expiries_synth = [today + timedelta(days=d) for d in [30, 60, 90]]
    strikes_synth = [int(INDEX_PRICE * m) for m in [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4]]
    contracts = [
        OptionContract(
            symbol=CURRENCY, expiry=exp, strike=Decimal(str(k)), right=Right(r),
            multiplier=1, exchange="DERIBIT", currency="USD",
        )
        for exp in expiries_synth for k in strikes_synth for r in ("C", "P")
    ]

by_expiry_map: dict[date, list[OptionContract]] = defaultdict(list)
for c in contracts:
    by_expiry_map[c.expiry].append(c)
expiries_sorted = sorted(by_expiry_map)

print(f"Instruments découverts : {len(contracts)}")
print(f"Expiries disponibles   : {len(expiries_sorted)}")

In [ ]:
# %% Plot 1 — Contracts per DTE
dtes   = [(exp - date.today()).days for exp in expiries_sorted]
counts = [len(by_expiry_map[exp]) for exp in expiries_sorted]
labels = [str(exp) for exp in expiries_sorted]

fig = go.Figure(go.Bar(
    x=dtes, y=counts,
    marker_color=C["blue"], marker_line_color=C["white"], marker_line_width=0.5,
    opacity=0.9,
    customdata=labels,
    hovertemplate="<b>%{customdata}</b><br>DTE: %{x}d<br>Contracts: %{y:,}<extra></extra>",
))
fig.update_layout(
    title=f"{CURRENCY} — Open Contracts by Days to Expiry",
    xaxis_title="DTE (days)", yaxis_title="Contract Count",
    bargap=0.25, height=350,
)
fig.show()

## 3 · Collecte des quotes REST → Snapshot

In [ ]:
# %% Sélection des expiries à capturer
N_EXPIRIES = min(3, len(expiries_sorted))
selected_expiries  = expiries_sorted[:N_EXPIRIES]
selected_contracts = [c for c in contracts if c.expiry in set(selected_expiries)]
print(f"Expiries sélectionnées : {selected_expiries}")
print(f"Contrats à quoter      : {len(selected_contracts)}")

In [ ]:
# %% Fetch des tickers + construction des RawMarketEvent
SESSION_ID = f"notebook-deribit-{datetime.now(UTC).strftime('%Y%m%d%H%M%S')}"
AS_OF = datetime.now(UTC)
events: list[RawMarketEvent] = []
_eid = 0

_DERIBIT_FIELDS = {
    "best_bid_price": "bid", "best_ask_price": "ask",
    "last_price": "last",    "mark_price": "mark_price",
}


def _eid_next() -> str:
    global _eid
    _eid += 1
    return str(_eid)


def _event(instrument_key_str: str, field: str, value: float | None) -> RawMarketEvent:
    return RawMarketEvent(
        collector_session_id=SESSION_ID, event_id=_eid_next(), receipt_ts=AS_OF,
        instrument_key=instrument_key_str, field_name=field,
        field_value=Decimal(str(value)) if value is not None else None,
        underlying=CURRENCY,
    )


btc_underlying = Underlying(symbol=CURRENCY, security_type="CRYPTO", exchange="DERIBIT", currency="USD")
btc_key = instrument_key(btc_underlying)
spread_half = INDEX_PRICE * 0.0001
for field, val in [("bid", INDEX_PRICE - spread_half), ("ask", INDEX_PRICE + spread_half), ("last", INDEX_PRICE)]:
    events.append(_event(btc_key, field, val))


def _deribit_name(c: OptionContract) -> str:
    expiry_s = f"{c.expiry.day}{c.expiry.strftime('%b%y').upper()}"
    strike_s = str(int(c.strike)) if c.strike == c.strike.to_integral_value() else str(c.strike)
    return f"{c.symbol}-{expiry_s}-{strike_s}-{c.right.value}"


def _synthetic_ticker(c: OptionContract, spot: float) -> dict:
    from scipy.stats import norm  # type: ignore[import-untyped]
    k = float(c.strike)
    t_yr = (c.expiry - date.today()).days / 365.0
    moneyness = k / spot
    iv = 0.70 + 0.15 * max(0.0, math.log(1 / moneyness)) + 0.05 * (moneyness - 1) ** 2
    if t_yr <= 0:
        return {"best_bid_price": None, "best_ask_price": None, "last_price": None, "mark_price": None, "mark_iv": iv * 100}
    fwd = spot
    d1 = (math.log(fwd / k) + 0.5 * iv**2 * t_yr) / (iv * math.sqrt(t_yr))
    d2 = d1 - iv * math.sqrt(t_yr)
    px = (fwd * norm.cdf(d1) - k * norm.cdf(d2)) if c.right == Right.CALL else (k * norm.cdf(-d2) - fwd * norm.cdf(-d1))
    px_btc = px / spot
    spread = max(0.0001, px_btc * 0.05)
    return {"best_bid_price": max(0.0001, px_btc - spread / 2), "best_ask_price": px_btc + spread / 2, "last_price": px_btc, "mark_price": px_btc, "mark_iv": iv * 100}


n_fetched, n_skipped = 0, 0
for c in selected_contracts:
    opt_key = instrument_key(c)
    if LIVE_DATA:
        try:
            ticker = transport.get("/public/ticker", {"instrument_name": _deribit_name(c)})
        except Exception as exc:  # noqa: BLE001
            print(f"  skip {_deribit_name(c)}: {exc}")
            n_skipped += 1
            continue
    else:
        ticker = _synthetic_ticker(c, INDEX_PRICE)
    for src, canon in _DERIBIT_FIELDS.items():
        raw = ticker.get(src)
        events.append(_event(opt_key, canon, round(float(raw) * INDEX_PRICE, 6) if raw is not None else None))
    raw_iv = ticker.get("mark_iv")
    if raw_iv is not None:
        events.append(_event(opt_key, "mark_iv", round(float(raw_iv) / 100.0, 6)))
    n_fetched += 1

print(f"Tickers collectes : {n_fetched}  |  ignores : {n_skipped}")
print(f"RawMarketEvents   : {len(events)}")

In [ ]:
# %% Construction du snapshot + filtrage QC
from algotrading.infra.qc import QCParams, filtered_snapshot, is_usable_quote, run_qc
from algotrading.infra.qc.state import QCStatus

snap_params = SnapshotParams(max_quote_age_seconds=300.0, max_spread_pct=Decimal("0.60"))
snapshot: MarketStateSnapshot = build_snapshot(events, AS_OF, snap_params)

qc_params  = QCParams.from_config(load_config(CONFIGS / "qc.yaml"))
qc_report  = run_qc(snapshot, qc_params)
clean      = filtered_snapshot(snapshot, qc_report, qc_params)

n_usable = sum(1 for q in snapshot.options if is_usable_quote(q))
print(f"Options dans snap : {len(snapshot.options)}")
print(f"Options usables   : {n_usable} / {len(snapshot.options)}  →  clean: {len(clean.options)}")

reject_reasons: dict[str, int] = {}
for v in qc_report.verdicts:
    if v.status is QCStatus.USABLE:
        continue
    for o in v.outcomes:
        if o.status is not QCStatus.USABLE and o.reason_code:
            reject_reasons[o.reason_code] = reject_reasons.get(o.reason_code, 0) + 1
if reject_reasons:
    print(f"Raisons de rejet QC : {dict(sorted(reject_reasons.items(), key=lambda kv: -kv[1]))}")

In [ ]:
# %% Plot 2 — Bid-ask spread % per strike (scatter) + distribution (histogram)
from algotrading.infra.universe.contracts import parse_instrument_key

strikes_all, spreads_all, is_usable_all = [], [], []
for q in snapshot.options:
    if q.bid is None or q.ask is None:
        continue
    c = parse_instrument_key(q.instrument_key)
    if not isinstance(c, OptionContract):
        continue
    mid = float(q.mid) if q.mid else None
    if mid and mid > 0:
        spread_rel = float(q.ask - q.bid) / mid
        strikes_all.append(float(c.strike))
        spreads_all.append(spread_rel * 100)
        is_usable_all.append(is_usable_quote(q))

strikes_arr = np.array(strikes_all)
spreads_arr = np.array(spreads_all)
usable_mask = np.array(is_usable_all)

fig = make_subplots(
    cols=2, column_widths=[0.78, 0.22], shared_yaxes=True,
    subplot_titles=("Spread % by Strike", "Distribution"),
)

for label, color, mask in [("Usable", C["green"], usable_mask), ("Rejected QC", C["red"], ~usable_mask)]:
    fig.add_trace(go.Scatter(
        x=strikes_arr[mask], y=spreads_arr[mask], mode="markers",
        marker=dict(color=color, size=6, opacity=0.75, line=dict(color=C["white"], width=0.5)),
        name=label,
        hovertemplate=f"<b>Strike %{{x:.1f}}</b><br>Spread: %{{y:.2f}}%<br>{label}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        y=spreads_arr[mask], nbinsy=30, marker_color=color, opacity=0.7,
        name=f"{label} dist.", showlegend=False,
    ), row=1, col=2)

spot_ref = float(snapshot.reference_spot)
fig.add_vline(x=spot_ref, line_dash="dash", line_color=C["slate400"], row=1, col=1)
fig.update_layout(
    title=f"{CURRENCY} — Bid-Ask Spread % by Strike",
    yaxis_title="Spread (%)", xaxis_title="Strike (USD)",
    height=380, barmode="overlay",
)
fig.show()

## 4 · Courbe forward (parité put-call)

In [ ]:
# %% Courbe forward
fwd_params = ForwardParams(
    max_candidate_count=12, max_robust_zscore=3.5, epsilon=Decimal("0.0001"),
    min_candidates=2, day_count=DayCountConvention("ACT/365F"), min_maturity_years=0.005,
    min_rate=Decimal("-0.10"), max_rate=Decimal("0.60"), comfortable_candidate_count=3,
)
forward_curve = build_forward_curve(clean, fwd_params)

print(f"Maturites avec forward accepte : {len(forward_curve.maturities)}")
for m in forward_curve.maturities:
    dte = int(m.maturity_years * 365)
    print(f"  {m.expiry}  ({dte:3d}j)  F={float(m.forward_price):,.0f}  r={float(m.rate)*100:+.2f}%  [{m.quality.value}]")

In [ ]:
# %% Plot 3 — Forward basis + implied carry (shared x-axis)
if not forward_curve.maturities:
    print("Aucun forward accepté — plot ignoré.")
else:
    spot_ref = float(snapshot.reference_spot)
    dtes_m  = [int(m.maturity_years * 365) for m in forward_curve.maturities]
    basis   = [float(m.forward_price) - spot_ref for m in forward_curve.maturities]
    rates   = [float(m.rate) for m in forward_curve.maturities]
    expiry_labels = [str(m.expiry) for m in forward_curve.maturities]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=("Forward Basis  (F − Spot, USD)", "Implied Carry Rate"),
        row_heights=[0.5, 0.5],
    )
    fig.add_trace(go.Scatter(
        x=dtes_m, y=basis, mode="lines+markers",
        line=dict(color=C["blue"], width=2), marker=dict(size=6),
        customdata=expiry_labels,
        hovertemplate="<b>%{customdata}</b><br>DTE: %{x}d<br>Basis: %{y:,.2f} USD<extra></extra>",
        name="F − Spot",
    ), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate400"], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=dtes_m, y=rates, mode="lines+markers",
        line=dict(color=C["teal"], width=2), marker=dict(size=6),
        customdata=expiry_labels,
        hovertemplate="<b>%{customdata}</b><br>DTE: %{x}d<br>Carry: %{y:.2%}<extra></extra>",
        name="Implied Carry",
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate400"], row=2, col=1)

    fig.update_yaxes(tickformat=",.0f", title_text="Basis (USD)", row=1, col=1)
    fig.update_yaxes(tickformat=".2%", title_text="Rate", row=2, col=1)
    fig.update_xaxes(title_text="DTE (days)", row=2, col=1)
    fig.update_layout(
        title=f"{CURRENCY} — Forward Structure", height=500,
    )
    fig.show()

## 5 · Inversion IV — chaîne d'options

In [ ]:
# %% Inversion IV
iv_params = IvParams.from_config(load_config(CONFIGS / "qc.yaml"))
iv_points = solve_chain(clean, forward_curve, iv_params)

n_solved   = sum(1 for p in iv_points if p.status == "solved")
n_unsolved = len(iv_points) - n_solved
print(f"IvPoints : {len(iv_points)}  →  solved: {n_solved}  unsolved: {n_unsolved}")

if n_unsolved:
    reasons: dict[str, int] = {}
    for p in iv_points:
        if p.status != "solved" and p.failure_reason:
            reasons[p.failure_reason] = reasons.get(p.failure_reason, 0) + 1
    print("  Raisons echec:", reasons)

In [ ]:
# %% Plot 4 — IV smile per expiry (calls solid, puts dotted, same color per expiry)
if not iv_points:
    print("Aucun IvPoint — plot ignoré.")
else:
    solved_pts = [p for p in iv_points if p.status == "solved" and p.implied_vol is not None]
    by_exp: dict = defaultdict(list)
    for p in solved_pts:
        by_exp[p.expiry_date].append(p)

    fig = go.Figure()
    for i, (exp, pts) in enumerate(sorted(by_exp.items())):
        color = DISCRETE[i % len(DISCRETE)]
        dte   = (exp - date.today()).days
        for side, right, dash in [("Call", Right.CALL, "solid"), ("Put", Right.PUT, "dot")]:
            side_pts = sorted(
                [p for p in pts if parse_instrument_key(p.contract_key).right == right],
                key=lambda p: p.log_moneyness,
            )
            if not side_pts:
                continue
            fig.add_trace(go.Scatter(
                x=[p.log_moneyness for p in side_pts],
                y=[p.implied_vol * 100 for p in side_pts],
                mode="lines+markers",
                line=dict(color=color, width=1.8, dash=dash),
                marker=dict(size=4),
                name=f"{exp} {side}",
                legendgroup=str(exp),
                legendgrouptitle_text=f"{exp} ({dte}d)" if side == "Call" else None,
                hovertemplate=(
                    f"<b>{exp} {side}</b><br>"
                    "log-m: %{x:.3f}<br>IV: %{y:.1f}%<extra></extra>"
                ),
            ))

    fig.add_vline(x=0, line_dash="dash", line_color=C["slate400"])
    fig.update_layout(
        title=f"{CURRENCY} — Implied Volatility Smile by Expiry",
        xaxis_title="Log-moneyness ln(K/F)", yaxis_title="IV (%)",
        legend=dict(groupclick="toggleitem"),
        height=420,
    )
    fig.show()

## 6 · Surface SVI

In [ ]:
# %% Surface SVI
from algotrading.infra.surfaces.svi import svi_total_variance

qc_cfg     = load_config(CONFIGS / "qc.yaml")
surface_cfg = SurfaceConfig.from_config(qc_cfg.data["surface_engine"])
solved_iv  = [p for p in iv_points if p.status == "solved"]
surface_result: SurfaceResult = build_surface(solved_iv, surface_cfg, config_hash=qc_cfg.config_hash)

print(f"Slices SVI calibrees : {len(surface_result.parameters.slices)}")
for sl in surface_result.parameters.slices:
    dte  = int(sl.maturity_years * 365)
    rmse = f"RMSE={sl.rmse:.5f}" if sl.rmse is not None else "RMSE=n/a"
    warn = "  [borne!]" if sl.bound_hits else ""
    print(f"  {dte:3d}j  [{sl.method.upper():5s}]  n={sl.n_points}  {rmse}{warn}")

diag = surface_result.diagnostics
print(f"\nViolations calendaires : {len(diag.calendar_violations)}")

In [ ]:
# %% Plot 5 — SVI fit vs market points per slice
slices   = surface_result.parameters.slices
raw_pts  = surface_result.raw_points

if not slices:
    print("Aucune slice — plot ignoré.")
else:
    ncols = 3
    nrows = (len(slices) + ncols - 1) // ncols
    dte_labels = [f"{int(sl.maturity_years * 365)}d  {sl.method.upper()}" for sl in slices]

    fig = make_subplots(
        rows=nrows, cols=ncols, shared_yaxes=True,
        horizontal_spacing=0.06, vertical_spacing=0.12,
        subplot_titles=dte_labels,
    )

    for idx, sl in enumerate(slices):
        row, col = idx // ncols + 1, idx % ncols + 1
        dte = int(sl.maturity_years * 365)

        raw = next((r for r in raw_pts if abs(r.maturity_years - sl.maturity_years) < 1e-4), None)
        if raw and raw.log_moneyness:
            rmse_s = f"RMSE={sl.rmse:.4f}" if sl.rmse is not None else ""
            warn   = " [bound hit]" if sl.bound_hits else ""
            fig.add_trace(go.Scatter(
                x=raw.log_moneyness, y=raw.total_variance, mode="markers",
                marker=dict(color=C["blue"], size=7, opacity=0.85, line=dict(color=C["white"], width=0.5)),
                name="Market" if idx == 0 else None, showlegend=(idx == 0),
                hovertemplate=f"<b>{dte}d Market</b><br>k: %{{x:.3f}}<br>w(k): %{{y:.5f}}<extra></extra>",
            ), row=row, col=col)

        if sl.method == "svi" and sl.params is not None:
            p      = sl.params
            k_grid = np.linspace(-0.5, 0.5, 200)
            tv_fit = svi_total_variance(k_grid, a=p.a, b=p.b, rho=p.rho, m=p.m, sigma=p.sigma)
            fig.add_trace(go.Scatter(
                x=k_grid, y=tv_fit, mode="lines",
                line=dict(color=C["red"], width=2),
                name="SVI fit" if idx == 0 else None, showlegend=(idx == 0),
                hovertemplate=f"<b>{dte}d SVI</b><br>k: %{{x:.3f}}<br>ŵ(k): %{{y:.5f}}<extra></extra>",
            ), row=row, col=col)

    fig.update_yaxes(tickformat=".4f")
    fig.update_layout(
        title=f"{CURRENCY} — SVI Calibration: Total Variance vs Log-Moneyness",
        height=280 * nrows,
    )
    fig.show()

In [ ]:
# %% Plot 6 — 3D IV Surface (interactive, frontend-ready)
grid = surface_result.grid

if grid is None or not grid.maturity_years:
    print("Grille indisponible — plot ignoré.")
else:
    k_axis = np.array(grid.log_moneyness)
    t_axis = np.array(grid.maturity_years)
    tv     = np.array(grid.total_variance)  # shape (n_maturities, n_moneyness)

    with np.errstate(divide="ignore", invalid="ignore"):
        iv_grid = np.where(t_axis[:, None] > 0, np.sqrt(np.maximum(tv, 0) / t_axis[:, None]), np.nan)

    # Build meshgrid: x=log-moneyness, y=maturity (years)
    kk, tt = np.meshgrid(k_axis, t_axis)

    fig = go.Figure()

    # ── Surface ──────────────────────────────────────────────────────────────
    fig.add_trace(go.Surface(
        x=kk, y=tt, z=iv_grid,
        colorscale=SURFACE_COLORSCALE,
        opacity=0.88,
        lighting=dict(ambient=0.7, diffuse=0.6, roughness=0.5, specular=0.1),
        lightposition=dict(x=1000, y=1000, z=1000),
        contours=dict(
            z=dict(show=True, usecolormap=True, highlightcolor=C["white"], project_z=True),
            x=dict(show=False),
            y=dict(show=False),
        ),
        colorbar=dict(
            title=dict(text="IV", side="right"),
            tickformat=".0%",
            thickness=16, len=0.7, x=1.02,
        ),
        hovertemplate=(
            "log-m: %{x:.3f}<br>"
            "Maturity: %{y:.3f}y<br>"
            "IV: %{z:.1%}<extra></extra>"
        ),
        name="IV Surface",
    ))

    # ── Calendar arbitrage violations ────────────────────────────────────────
    viol = surface_result.diagnostics.calendar_violations
    if viol:
        vx = [v.log_moneyness for v in viol]
        vy = [v.maturity_high for v in viol]
        vz = [np.sqrt(max(v.variance_high, 0.0) / v.maturity_high) for v in viol]
        fig.add_trace(go.Scatter3d(
            x=vx, y=vy, z=vz,
            mode="markers",
            marker=dict(color=C["red"], size=6, symbol="x", line=dict(color=C["white"], width=1)),
            hovertemplate=(
                "<b>CALENDAR VIOLATION</b><br>"
                "log-m: %{x:.3f}<br>Maturity: %{y:.3f}y<br>IV: %{z:.1%}<extra></extra>"
            ),
            name="Calendar violation",
        ))

    fig.update_layout(
        title=f"{CURRENCY} — Implied Volatility Surface (SVI)",
        height=640,
        scene=dict(
            xaxis=dict(
                title="Log-Moneyness",
                gridcolor=C["slate400"], backgroundcolor=C["slate100"], showbackground=True,
            ),
            yaxis=dict(
                title="Maturity (yr)",
                gridcolor=C["slate400"], backgroundcolor=C["slate100"], showbackground=True,
            ),
            zaxis=dict(
                title="IV", tickformat=".0%",
                gridcolor=C["slate400"], backgroundcolor=C["slate100"], showbackground=True,
            ),
            camera=SURFACE_CAMERA,
            aspectmode="manual",
            aspectratio=SURFACE_ASPECT,
        ),
        margin=dict(l=0, r=0, t=56, b=0),
    )
    fig.show()

    # Sérialisabilité frontend — valider que fig.to_json() fonctionne
    import json
    _json_bytes = len(fig.to_json().encode())
    print(f"✓ fig.to_json() OK — {_json_bytes / 1024:.1f} KB (prêt pour FastAPI → react-plotly.js)")

## 7 · Pricing & Greeks sur la chaîne

In [ ]:
# %% Pricing + Greeks
pricing_params = PricingParams.from_config(load_config(CONFIGS / "pricing.yaml"))

solved_by_exp: dict = defaultdict(list)
for p in iv_points:
    if p.status == "solved" and p.implied_vol is not None:
        solved_by_exp[p.expiry_date].append(p)

_grid = surface_result.grid


def _surface_vol(k: float, t_yr: float, fallback: float) -> float:
    if _grid is None or t_yr <= 0:
        return fallback
    tv = _grid.total_variance_at(maturity=t_yr, k=k)
    return float(np.sqrt(max(tv, 0.0) / t_yr))


if not solved_by_exp:
    print("Aucun IvPoint solved — pricing impossible.")
else:
    first_exp = sorted(solved_by_exp)[0]
    pts_exp   = solved_by_exp[first_exp]
    spot      = float(snapshot.reference_spot)
    fwd_mat   = next((m for m in forward_curve.maturities if m.expiry == first_exp), None)
    forward   = float(fwd_mat.forward_price) if fwd_mat else spot
    rate      = float(fwd_mat.rate) if fwd_mat else 0.0
    carry     = float(fwd_mat.implied_carry) if fwd_mat else 0.0
    T         = pts_exp[0].maturity_years

    pricing_inputs: dict[str, PricingInput] = {}
    pricing_vol: dict[str, float] = {}
    for p in pts_exp:
        c = parse_instrument_key(p.contract_key)
        if not isinstance(c, OptionContract):
            continue
        vol = _surface_vol(p.log_moneyness, T, p.implied_vol)
        pricing_vol[p.contract_key] = vol
        pricing_inputs[p.contract_key] = PricingInput(
            spot=spot, strike=float(c.strike), maturity=T, vol=vol, rate=rate, carry=carry,
            is_call=(c.right == Right.CALL), multiplier=c.multiplier, style=OptionStyle.EUROPEAN,
        )

    results = {key: price(inp, pricing_params) for key, inp in pricing_inputs.items()}
    print(f"Expiry pricée : {first_exp} ({int(T * 365)}j)  —  {len(results)} options")

In [ ]:
# %% Plot 7 — Greeks 2×2 grid (price, delta, gamma, vega vs strike)
if not solved_by_exp:
    print("Pas de données — plot ignoré.")
else:
    calls_data = sorted(
        [(parse_instrument_key(k), results[k]) for k in results
         if isinstance(parse_instrument_key(k), OptionContract) and parse_instrument_key(k).right == Right.CALL],
        key=lambda x: float(x[0].strike),
    )
    puts_data = sorted(
        [(parse_instrument_key(k), results[k]) for k in results
         if isinstance(parse_instrument_key(k), OptionContract) and parse_instrument_key(k).right == Right.PUT],
        key=lambda x: float(x[0].strike),
    )

    greeks_cfg = [
        ("price", "Option Price",  ".2f",  C["blue"],   C["teal"],   (1, 1)),
        ("delta", "Delta Δ",       ".3f",  C["blue"],   C["teal"],   (1, 2)),
        ("gamma", "Gamma Γ",       ".5f",  C["violet"], C["amber"],  (2, 1)),
        ("vega",  "Vega",          ".2f",  C["violet"], C["amber"],  (2, 2)),
    ]

    fig = make_subplots(
        rows=2, cols=2, shared_xaxes=True,
        vertical_spacing=0.08, horizontal_spacing=0.08,
        subplot_titles=[g[1] for g in greeks_cfg],
    )

    for field, label, fmt, c_call, c_put, (row, col) in greeks_cfg:
        for side, data, color, dash in [("Call", calls_data, c_call, "solid"), ("Put", puts_data, c_put, "dot")]:
            xs = [float(c.strike) for c, _ in data]
            ys = [getattr(r, field) for _, r in data]
            fig.add_trace(go.Scatter(
                x=xs, y=ys, mode="lines",
                line=dict(color=color, width=2, dash=dash),
                name=side,
                legendgroup=side, showlegend=(field == "price"),
                hovertemplate=f"Strike %{{x:.1f}}<br>{label}: %{{y:{fmt}}}<extra></extra>",
            ), row=row, col=col)
        fig.update_yaxes(tickformat=fmt, row=row, col=col)

    fig.add_vline(x=spot, line_dash="dash", line_color=C["slate400"])
    fig.update_xaxes(title_text="Strike (USD)", row=2, col=1)
    fig.update_xaxes(title_text="Strike (USD)", row=2, col=2)
    fig.update_layout(
        title=f"{CURRENCY} {first_exp} ({int(T*365)}d) — Greeks vs Strike  (calls solid, puts dotted)",
        height=560,
    )
    fig.show()

## 8 · Risque — portefeuille fictif

In [ ]:
# %% Risque — straddle ATM fictif
risk_lines: list = []
agg = atm_call_key = atm_put_key = None

if not solved_by_exp or not results:
    print("Pas de données pricées — risk ignoré.")
else:
    lm = {p.contract_key: p.log_moneyness for p in solved_by_exp[first_exp]}
    atm_key    = min(results, key=lambda k: abs(lm.get(k, 999.0)))
    atm_strike = parse_instrument_key(atm_key).strike

    def _leg(right: Right) -> str | None:
        for k in results:
            c = parse_instrument_key(k)
            if c.right == right and c.strike == atm_strike:
                return k
        return None

    atm_call_key = _leg(Right.CALL)
    atm_put_key  = _leg(Right.PUT)

    if not atm_call_key or not atm_put_key:
        print("Call ou put ATM introuvable — risk ignoré.")
    else:
        positions = PositionSet(
            positions=(
                Position(instrument_key=atm_call_key, quantity=Decimal("1")),
                Position(instrument_key=atm_put_key,  quantity=Decimal("1")),
            ),
            source="notebook-demo", source_ts=AS_OF,
        )
        analytics = {
            key: InstrumentAnalytics(pricing=results[key], multiplier=parse_instrument_key(key).multiplier)
            for key in (atm_call_key, atm_put_key)
        }
        risk_lines = compute_risk_lines(positions, analytics)
        agg        = aggregate_risk(risk_lines, by_underlying)[0]
        theta_unit = pricing_params.theta_unit
        theta_suffix = "/yr" if theta_unit == "year" else "/d"

        print(f"Strike ATM : K={float(atm_strike):,.0f}")
        print(f"Valeur totale : ${agg.value:.2f}  Delta: {agg.delta:+.4f}  Vega: ${agg.dollar_vega:+.2f}")

In [ ]:
# %% Plot 8 — Risk Greeks per leg (horizontal bars)
if not risk_lines or agg is None:
    print("Pas de données risk — plot ignoré.")
else:
    # Build leg labels + include net row
    leg_labels = []
    for line in risk_lines:
        c = parse_instrument_key(line.instrument_key)
        leg_labels.append(f"{c.right.value} K={float(c.strike):,.0f}")
    leg_labels.append("NET")

    greeks_risk = {
        "Delta":  [line.delta for line in risk_lines] + [agg.delta],
        "Gamma":  [line.gamma for line in risk_lines] + [agg.gamma],
        "Vega":   [line.dollar_vega for line in risk_lines] + [agg.dollar_vega],
        "Theta":  [line.theta for line in risk_lines] + [agg.theta],
    }
    greek_colors = {"Delta": C["blue"], "Gamma": C["violet"], "Vega": C["teal"], "Theta": C["amber"]}

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=list(greeks_risk.keys()),
        horizontal_spacing=0.12, vertical_spacing=0.15,
    )
    positions_map = [(1, 1), (1, 2), (2, 1), (2, 2)]

    for (greek, vals), (row, col) in zip(greeks_risk.items(), positions_map):
        base_color = greek_colors[greek]
        bar_colors = [C["red"] if v < 0 else base_color for v in vals]
        fig.add_trace(go.Bar(
            y=leg_labels, x=vals, orientation="h",
            marker_color=bar_colors,
            name=greek, showlegend=False,
            hovertemplate=f"<b>%{{y}}</b><br>{greek}: %{{x:.4f}}<extra></extra>",
        ), row=row, col=col)
        fig.update_xaxes(
            zeroline=True, zerolinecolor=C["slate400"], zerolinewidth=1.5, row=row, col=col,
        )

    fig.update_layout(
        title=f"{CURRENCY} Straddle ATM — Greeks by Leg (red = negative)",
        height=460,
    )
    fig.show()

## 9 · Scénarios de stress

In [ ]:
# %% Scénarios de stress
if not solved_by_exp or not results or not risk_lines:
    print("Pas de données — scénarios ignorés.")
else:
    scenario_grid = ScenarioGrid(
        scenarios=(
            Scenario(name="spot_up_10",        family="spot",  spot_shock=0.10,  vol_shock=0.00, time_shock_days=0),
            Scenario(name="spot_up_20",        family="spot",  spot_shock=0.20,  vol_shock=0.00, time_shock_days=0),
            Scenario(name="spot_down_10",      family="spot",  spot_shock=-0.10, vol_shock=0.00, time_shock_days=0),
            Scenario(name="spot_down_20",      family="spot",  spot_shock=-0.20, vol_shock=0.00, time_shock_days=0),
            Scenario(name="vol_up_10pt",       family="vol",   spot_shock=0.00,  vol_shock=0.10, time_shock_days=0),
            Scenario(name="vol_down_10pt",     family="vol",   spot_shock=0.00,  vol_shock=-0.10, time_shock_days=0),
            Scenario(name="crash_down20_vol30",family="crash", spot_shock=-0.20, vol_shock=0.30, time_shock_days=0),
            Scenario(name="rally_up20_vol_dn", family="rally", spot_shock=0.20,  vol_shock=-0.10, time_shock_days=0),
            Scenario(name="roll_1d",           family="time",  spot_shock=0.00,  vol_shock=0.00, time_shock_days=1),
            Scenario(name="roll_7d",           family="time",  spot_shock=0.00,  vol_shock=0.00, time_shock_days=7),
        ),
        version="notebook-btc-v1",
    )
    states = {key: pricing_inputs[key] for key in (atm_call_key, atm_put_key) if key in pricing_inputs}
    scenario_results = run_grid(positions, states, pricing_params, scenario_grid)
    print(f"Scénarios exécutés : {len(scenario_results)}")

In [ ]:
# %% Plot 9 — Stress PnL: scatter full vs local + grouped bar per scenario
if not solved_by_exp or not results or not risk_lines:
    print("Pas de données — plot ignoré.")
else:
    names  = [r.scenario for r in scenario_results]
    full   = np.array([r.full_pnl for r in scenario_results])
    local  = np.array([r.local_pnl for r in scenario_results])
    abs_err = np.abs(full - local)
    families = [r.family for r in scenario_results]
    family_color_map = {"spot": C["blue"], "vol": C["teal"], "crash": C["red"], "rally": C["green"], "time": C["amber"]}

    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.50, 0.50],
        subplot_titles=("Full Reprice vs Greeks Approx", "PnL by Scenario"),
    )

    # Left — scatter full vs local, color = abs error
    fig.add_trace(go.Scatter(
        x=full, y=local, mode="markers",
        marker=dict(
            color=abs_err,
            colorscale=[[0, C["green"]], [0.5, C["amber"]], [1, C["red"]]],
            colorbar=dict(title="Abs Error", x=0.47, thickness=12, len=0.8),
            size=9, opacity=0.9, line=dict(color=C["white"], width=0.5),
        ),
        customdata=names,
        hovertemplate="<b>%{customdata}</b><br>Full: %{x:.2f}<br>Greeks: %{y:.2f}<br>Err: %{marker.color:.2f}<extra></extra>",
        showlegend=False,
    ), row=1, col=1)

    pnl_range = [float(min(np.min(full), np.min(local))), float(max(np.max(full), np.max(local)))]
    fig.add_trace(go.Scatter(
        x=pnl_range, y=pnl_range, mode="lines",
        line=dict(color=C["slate400"], dash="dash", width=1),
        name="Perfect fit", showlegend=True,
    ), row=1, col=1)

    # Right — grouped bar: full repricing
    fig.add_trace(go.Bar(
        x=names, y=full,
        marker_color=[C["green"] if v >= 0 else C["red"] for v in full],
        name="Full Reprice",
        hovertemplate="<b>%{x}</b><br>Full PnL: %{y:.2f}<extra></extra>",
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=names, y=local,
        marker_color=C["blue"], opacity=0.6,
        marker_pattern_shape="/",
        name="Greeks Approx",
        hovertemplate="<b>%{x}</b><br>Greeks PnL: %{y:.2f}<extra></extra>",
    ), row=1, col=2)

    fig.update_xaxes(title_text="Full Reprice PnL", row=1, col=1)
    fig.update_yaxes(title_text="Greeks Approx PnL", row=1, col=1)
    fig.update_xaxes(tickangle=30, row=1, col=2)
    fig.update_yaxes(title_text="PnL (USD)", row=1, col=2)
    fig.update_layout(
        title=f"{CURRENCY} Straddle ATM — Stress PnL Analysis",
        barmode="group", height=460,
    )
    fig.show()

## Récapitulatif de la pile

| Étape | Module | Entrée → Sortie |
|-------|--------|------------------|
| 1 Connexion | `infra_deribit.connectivity` | REST → `index_price` |
| 2 Discovery | `infra_deribit.collectors.deribit_discovery` | REST → `list[OptionContract]` |
| 3 Snapshot | `infra.snapshots` | `list[RawMarketEvent]` → `MarketStateSnapshot` |
| 4 Forwards | `infra.forwards` | snapshot → `ForwardCurve` (parité put-call) |
| 5 IV chain | `infra.iv` | snapshot + curve → `list[IvPoint]` |
| 6 Surface SVI | `infra.surfaces` | IvPoints → `SurfaceResult` (slices + grille) |
| 7 Pricing | `infra.pricing` | `PricingInput` → `PricingResult` (prix + Greeks) |
| 8 Risk | `infra.risk` | positions + analytics → `RiskLine[]` + agrégats |
| 9 Scénarios | `infra.risk.scenarios` | positions + états → `ScenarioResult[]` |

**Surface 3D frontend-ready** : `fig.to_json()` produit un objet Plotly directement consommable par `react-plotly.js` — aucune transformation côté React.